In [1]:
from geneticengine.grammar.decorators import abstract
from geneticengine.grammar.grammar import extract_grammar
from geneticengine.algorithms.gp.gp import GeneticProgramming
from geneticengine.random.sources import NativeRandomSource
from geneticengine.representations.tree.initializations import MaxDepthDecider
from geneticengine.representations.tree.treebased import TreeBasedRepresentation
from geneticengine.problems import SingleObjectiveProblem
from geneticengine.evaluation.budget import EvaluationBudget, TimeBudget
from geneticengine.grammar.metahandlers.ints import IntRange
from geneticengine.evaluation.recorder import CSVSearchRecorder
from geneticengine.evaluation.tracker import ProgressTracker


from abc import ABC
from dataclasses import dataclass

import numpy as np

from sklearn.datasets import load_iris, load_wine
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn import metrics
from typing import Annotated

In [2]:
wine = load_wine()
X, y = wine.data, wine.target

In [3]:
#core classes
class Feature(ABC):
    def evaluate(self, X):
        pass

@dataclass
class PrimitiveFeature(Feature):
    column_index: Annotated[int, IntRange(0,12)]

    def evaluate(self, X):
        return X[:, self.column_index]
    
    def __str__(self):
        return f"column_{wine.feature_names[self.column_index]}"
        
@dataclass
class Add(Feature):
    left : Feature #type feature
    right : Feature #type feature
    def evaluate(self, X):
        return self.left.evaluate(X) + self.right.evaluate(X)
    def __str__(self):
        return f"({self.left} + {self.right})"

@dataclass
class Subtract(Feature):
    left : Feature
    right : Feature
    def evaluate(self, X):
        return self.left.evaluate(X) - self.right.evaluate(X)
    def __str__(self):
        return f"({self.left} - {self.right})"


In [4]:
#grammar
components = [
    PrimitiveFeature,
    Add,
    Subtract
]

grammar = extract_grammar(components, Feature)
print(grammar)

Grammar<Starting=Feature,Productions={
Feature -> PrimitiveFeature(column_index: Annotated[int])|
	Add(left: Feature, right: Feature)|
	Subtract(left: Feature, right: Feature)
}


In [5]:
#fitness function
def fitness_function(feature: Feature) -> float:
    feature_values = feature.evaluate(X)

    X_combined = np.hstack([X, feature_values.reshape(-1,1)])
    
    clf = LogisticRegression(random_state=0, max_iter=200, solver='liblinear')
    f1 = cross_val_score(clf, X_combined, y, cv=3, scoring='f1_macro').mean()

    return f1

In [6]:
#GP setup
rnd = NativeRandomSource(123)
decider = MaxDepthDecider(rnd, grammar, max_depth=5)
representation = TreeBasedRepresentation(grammar, decider)
objective = SingleObjectiveProblem(fitness_function=fitness_function, minimize=False)

gp = GeneticProgramming(
    problem=objective,
    budget=TimeBudget(30),
    representation=representation,
    random=rnd,
    tracker= ProgressTracker(
        objective,
        recorders=[CSVSearchRecorder(csv_path='../../gp_outputs/test.csv', problem=objective)]
    ),
    population_size=50,
)

best = gp.search()[0]

In [7]:
print(f"{best.get_fitness(gp.get_problem())}\n{best.get_phenotype()}")

[0.9619335855758028]
((column_hue - ((column_flavanoids + column_color_intensity) + column_od280/od315_of_diluted_wines)) + (((column_alcalinity_of_ash - column_od280/od315_of_diluted_wines) - (column_malic_acid - column_magnesium)) - ((column_total_phenols - column_alcohol) + (column_total_phenols - column_total_phenols))))
